# Grill Predictions
Goal: Create a model, which can predict the estimated remaining time until a target temperature is reached.

## Generate training data

In [1]:
# grill_data_generator.py
import numpy as np
import pandas as pd

np.random.seed(42)

def simulate_grilling_session(
    target_temp:   float,
    ambient_start: float = 20.0,
    noise_std:     float = 0.5,
    n_points:      int   = None,
) -> pd.DataFrame:
    """
    Simulates a realistic grilling temperature curve.
    Temperature rises fast at first, then slows near target
    (Newton's law of cooling in reverse).
    """
    # Random session parameters
    total_minutes = np.random.uniform(20, 90)
    if n_points is None:
        n_points = int(total_minutes * 2)  # reading every 30s

    times = np.linspace(0, total_minutes, n_points)

    # Asymptotic rise toward target (realistic heat transfer model)
    # temp(t) = target - (target - start) * exp(-k * t)
    k     = np.random.uniform(0.03, 0.08)  # heat transfer coefficient
    temps = target_temp - (target_temp - ambient_start) * np.exp(-k * times)

    # Add realistic noise
    temps += np.random.normal(0, noise_std, n_points)

    return pd.DataFrame({
        'session_id':    [f"session_{np.random.randint(10000)}"] * n_points,
        'meat_type':     [np.random.choice(['beef', 'pork', 'chicken'])] * n_points,
        'target_temp':   [target_temp] * n_points,
        'time_elapsed':  times,
        'temperature':   temps,
        'time_remaining': total_minutes - times,
    })


# Generate many sessions with various targets
sessions = []
for _ in range(200):
    target = np.random.uniform(60, 85)   # rare to well done range
    session = simulate_grilling_session(target)
    # Only keep rows where we haven't reached target yet
    session = session[session['temperature'] < session['target_temp']]
    sessions.append(session)

df = pd.concat(sessions, ignore_index=True)
df.to_csv('grill_sessions.csv', index=False)
print(f"Generated {len(df)} readings across {len(sessions)} sessions")
print(df.head())
print(df.describe())

Generated 21443 readings across 200 sessions
     session_id meat_type  target_temp  time_elapsed  temperature  \
0  session_3354   chicken    69.363503      0.000000    19.444060   
1  session_3354   chicken    69.363503      0.503198    21.786348   
2  session_3354   chicken    69.363503      1.006395    23.339696   
3  session_3354   chicken    69.363503      1.509593    25.226859   
4  session_3354   chicken    69.363503      2.012791    25.902447   

   time_remaining  
0       86.550001  
1       86.046804  
2       85.543606  
3       85.040408  
4       84.537211  
        target_temp  time_elapsed   temperature  time_remaining
count  21443.000000  21443.000000  21443.000000    21443.000000
mean      73.484597     30.218637     56.127723       30.915764
std        6.863684     20.394908     15.588395       20.721677
min       60.087617      0.000000     18.832035        0.000000
25%       67.728357     13.287810     45.497945       13.771463
50%       73.352542     27.263445   

## EDA (Exploratory Data Analysis)

In [5]:
# eda.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('grill_sessions.csv')

# ── 1. Basic overview ─────────────────────────────────────────────────
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nBasic stats:\n", df.describe())

# ── 2. Distribution of target variable ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['time_remaining'], bins=50, edgecolor='black')
axes[0].set_title('Distribution of time_remaining (target variable)')
axes[0].set_xlabel('Minutes remaining')
axes[0].set_ylabel('Count')

# Log transform – time_remaining is right-skewed
axes[1].hist(np.log1p(df['time_remaining']), bins=50, edgecolor='black')
axes[1].set_title('Log distribution of time_remaining')
axes[1].set_xlabel('log(minutes remaining + 1)')
plt.tight_layout()
plt.savefig('eda_target_distribution.png')
plt.show()

# ── 3. Temperature curves – do they look realistic? ──────────────────
fig, ax = plt.subplots(figsize=(12, 6))
sample_sessions = df['session_id'].unique()[:10]
for sid in sample_sessions:
    s = df[df['session_id'] == sid]
    ax.plot(s['time_elapsed'], s['temperature'], alpha=0.6)
ax.set_xlabel('Time elapsed (minutes)')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Sample grilling curves')
plt.savefig('eda_curves.png')
plt.show()

# ── 4. Key insight: heating rate changes over time ────────────────────
# Compute per-session heating rates
df = df.sort_values(['session_id', 'time_elapsed'])
df['temp_diff'] = df.groupby('session_id')['temperature'].diff()
df['time_diff'] = df.groupby('session_id')['time_elapsed'].diff()
df['heating_rate'] = df['temp_diff'] / df['time_diff']

fig, ax = plt.subplots(figsize=(12, 6))
for sid in sample_sessions:
    s = df[df['session_id'] == sid].dropna()
    ax.plot(s['time_elapsed'], s['heating_rate'].rolling(3).mean(), alpha=0.6)
ax.set_xlabel('Time elapsed (minutes)')
ax.set_ylabel('Heating rate (°C/min)')
ax.set_title('Heating rate decelerates over time (key feature!)')
ax.axhline(y=0, color='red', linestyle='--')
plt.savefig('eda_heating_rate.png')
plt.show()

# ── 5. Correlation with target variable ──────────────────────────────
# What actually predicts time_remaining?
numeric_cols = ['temperature', 'target_temp', 'time_elapsed',
                'heating_rate', 'time_remaining']
corr = df[numeric_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Feature correlations')
plt.tight_layout()
plt.savefig('eda_correlations.png')
plt.show()

# Key findings to look for:
# - time_elapsed should correlate negatively with time_remaining (obvious)
# - heating_rate should correlate negatively with time_remaining
# - temp_remaining (target - current) should correlate strongly positively
print("\nCorrelation with time_remaining:")
print(corr['time_remaining'].sort_values(ascending=False))


Correlation with time_remaining:
time_remaining    1.000000
target_temp       0.034507
heating_rate     -0.015742
temperature      -0.519953
time_elapsed     -0.600230
Name: time_remaining, dtype: float64


# Feature Engineering

In [6]:
# features.py
import pandas as pd
import numpy as np

def engineer_features(df: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    """
    Create features for each reading based on session history so far.
    window = number of recent readings to compute recent_rate from.
    """
    df = df.sort_values(['session_id', 'time_elapsed']).copy()

    # Per-reading features
    df['temp_remaining'] = df['target_temp'] - df['temperature']

    # Rate features – computed per session
    df['temp_diff'] = df.groupby('session_id')['temperature'].diff()
    df['time_diff'] = df.groupby('session_id')['time_elapsed'].diff()

    # fillna(0) instead of leaving NaN for first row of each session
    df['inst_rate'] = (df['temp_diff'] / df['time_diff'].replace(0, np.nan)).fillna(0)

    # Average rate from session start
    df['avg_rate'] = df.groupby('session_id').apply(
        lambda g: g['temperature'].diff().fillna(0).cumsum() / g['time_elapsed'].clip(lower=0.01)
    ).reset_index(level=0, drop=True)

    # Recent rate – average over last N readings
    df['recent_rate'] = df.groupby('session_id')['inst_rate'] \
        .transform(lambda x: x.rolling(window, min_periods=1).mean())

    # Rate trend – is heating accelerating or decelerating?
    df['rate_trend'] = df.groupby('session_id')['inst_rate'] \
        .transform(lambda x: x.rolling(window, min_periods=2).apply(
            lambda v: np.polyfit(range(len(v)), v, 1)[0] if len(v) > 1 else 0
        ))

    # Progress ratio – how far through the cook are we?
    df['progress_ratio'] = 1 - (df['temp_remaining'] / (df['target_temp'] - 20).clip(lower=1))

    # Drop rows without enough history
    df = df.dropna(subset=['recent_rate', 'avg_rate'])
    df = df[df['time_elapsed'] > 1]  # need at least 1 minute of data

    return df

FEATURE_COLS = [
    'temperature',      # current temp
    'target_temp',      # desired final temp
    'temp_remaining',   # how far to go
    'time_elapsed',     # how long grilling so far
    'avg_rate',         # overall heating rate
    'recent_rate',      # recent heating rate
    'rate_trend',       # accelerating or decelerating?
    'progress_ratio',   # 0-1, how close to done
]

TARGET_COL = 'time_remaining'

# Train and export to ONNX

In [8]:
# train.py
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

#from features import engineer_features, FEATURE_COLS, TARGET_COL

# ── 1. Load and prepare data ──────────────────────────────────────────
df_raw = pd.read_csv('grill_sessions.csv')
df     = engineer_features(df_raw)

# drop rows with NaN in any feature or target column
cols_needed = FEATURE_COLS + [TARGET_COL, 'session_id']
df = df.dropna(subset=cols_needed)
# also drop rows where rates are infinite (division by zero edge case)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=cols_needed)
print(f"Rows after dropping NaN: {len(df)}")

X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET_COL].values.astype(np.float32)

print(f"Training samples: {len(X)}")
print(f"Features: {FEATURE_COLS}")
print(f"Target range: {y.min():.1f} – {y.max():.1f} minutes")

# ── 2. Train/test split ───────────────────────────────────────────────
# Split by session, not by row – to avoid data leakage
# (rows from the same session should not be in both train and test)
sessions   = df['session_id'].unique()
train_sess, test_sess = train_test_split(sessions, test_size=0.2, random_state=42)

train_mask = df['session_id'].isin(train_sess)
test_mask  = df['session_id'].isin(test_sess)

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# ── 3. Train model ────────────────────────────────────────────────────
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  GradientBoostingRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42,
    )),
])

pipeline.fit(X_train, y_train)

# ── 4. Evaluate ───────────────────────────────────────────────────────
y_pred = pipeline.predict(X_test)
mae    = mean_absolute_error(y_test, y_pred)
r2     = r2_score(y_test, y_pred)

print(f"\nTest MAE:  {mae:.2f} minutes")
print(f"Test R²:   {r2:.3f}")

# Cross-validation
cv_scores = cross_val_score(
    pipeline, X_train, y_train,
    cv=5, scoring='neg_mean_absolute_error'
)
print(f"CV MAE:    {-cv_scores.mean():.2f} ± {cv_scores.std():.2f} minutes")

# Feature importance
importances = pipeline.named_steps['model'].feature_importances_
for feat, imp in sorted(zip(FEATURE_COLS, importances),
                        key=lambda x: x[1], reverse=True):
    print(f"  {feat:20s}: {imp:.3f}")

# ── 5. Export to ONNX ─────────────────────────────────────────────────
initial_type = [("float_input", FloatTensorType([None, len(FEATURE_COLS)]))]

onnx_model = convert_sklearn(
    pipeline,
    initial_types=initial_type,
    target_opset={"": 17, "ai.onnx.ml": 3},
)

with open("grill_predictor.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

# Save feature metadata for Rust
metadata = {
    "feature_cols":  FEATURE_COLS,
    "target_col":    TARGET_COL,
    "n_features":    len(FEATURE_COLS),
    "training_mae":  float(mae),
    "training_r2":   float(r2),
}
with open("grill_predictor_meta.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nExported grill_predictor.onnx")

TypeError: unsupported format string passed to numpy.ndarray.__format__

In [27]:
# ── 6. Verify with realistic test case ───────────────────────────────
import onnxruntime as rt

sess = rt.InferenceSession("grill_predictor.onnx")

# Simulated: 20 min in, beef, target 68°C, currently at 50°C
test_input = np.array([[
    50.0,   # temperature
    68.0,   # target_temp
    18.0,   # temp_remaining
    20.0,   # time_elapsed
    1.2,    # avg_rate
    0.8,    # recent_rate (slowing down)
    -0.05,  # rate_trend (decelerating)
    0.625,  # progress_ratio
]], dtype=np.float32)

pred = sess.run(None, {"float_input": test_input})
min_remaining = pred[0][0]
min_remaining
#print(f"\nTest prediction: {pred[0][0]:.1f} minutes remaining")

array([19.347706], dtype=float32)

In [15]:
print(f"\nTest prediction: {min_remaining:.1f} minutes remaining")

TypeError: unsupported format string passed to numpy.ndarray.__format__